In [ ]:
import numpy as np
import h5py
import hdf5plugin
import matplotlib.pyplot as plt
from matplotlib import colors
from pathlib import Path
from collections import OrderedDict
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.animation import FuncAnimation
from collections import OrderedDict
from IPython.display import HTML,display
from scipy import ndimage
from scipy.ndimage import generic_filter
import re
from scipy.signal import find_peaks, peak_widths

In [ ]:
# adjust these to whatever you require for your live analysis
#h5_loc=Path(r'X:\bl17-2\2026_06_Lindenberg\BTO_001')
#h5_loc=Path(r'C:\Users\leeau\Dropbox\Stanford\Lindenberg Lab\2026_06_Lindenberg\BTO_001')
save_path = #Path(r'C:\Users\leeau\Dropbox\Stanford\Lindenberg Lab\2026_06_Lindenberg\analysis/BTO001_002')
#waveform_path = Path(r'C:\Users\leeau\Dropbox\Stanford\Lindenberg Lab\2026_06_Lindenberg\ssrl_bto\post_run_gate_locations')
#current_path = Path(r'C:\Users\leeau\Dropbox\Stanford\Lindenberg Lab\2026_06_Lindenberg\ssrl_bto\Yuejun_current_tracings')

In [ ]:
#check path
h5_pattern='run*data*.h5'
h5_pattern.split("*", 1)[1]

In [ ]:
def animate_diff_images(results, x_param='eta', roi=None, 
                       key_ranges=None, vmin=-1000, vmax=1000, cmap='bwr',
                       show_com=True, 
                       use_absolute_pixels=True, 
                       xlim=None, ylim=None,
                       interval=500, 
                       figsize=(8, 6), save_path=None, save_html=False,
                       display_animation=True,
                       log_scale = False,
                       linthresh = 1.0): #threshold for symlognorm
    """
    Animate difference images from process_scan_data results.
    
    Parameters:
    -----------
    results : dict
        Results dictionary from process_scan_data containing:
        - 'diff_stack': OrderedDict of difference images
        - 'eta_dict', 'delay_dict', 'voltage_dict': Parameter values
        - 'com_g1', 'com_g2': Centers of mass (if show_com=True)
    x_param : str, optional
        Parameter to animate over: 'eta', 'delay', 'voltage', or 'run'
        Default: 'eta'
    roi : list, optional
        ROI used in processing [row_start, row_end, col_start, col_end]
        Needed if show_com=True to get relative coordinates
    key_ranges : list of tuples, optional
        List of (min, max) bounds for x_param to create separate animations
        Example: [(11.0, 11.5), (11.6, 12.0)]
    vmin, vmax : float, optional
        Color scale limits (default: -1000, 1000)
    cmap : str, optional
        Colormap name (default: 'bwr')
    show_com : bool, optional
        Whether to show center of mass markers (default: True)
    interval : int, optional
        Time between frames in milliseconds (default: 500)
    figsize : tuple, optional
        Figure size (default: (8, 6))
    save_path : str, optional
        Base filename for saving animations (.gif, .mp4, or .html if save_html=True)
    save_html : bool, optional
        If True, saves interactive HTML with slider instead of video (default: False)
    display_animation : bool, optional
        If True, displays the animation in notebook (default: True)
    
    Returns:
    --------
    animation_widgets : list
        List of HTML animation widgets
    """
    import matplotlib.pyplot as plt
    import matplotlib.colors as colors
    from matplotlib.animation import FuncAnimation
    from IPython.display import HTML, display
    from pathlib import Path
    import numpy as np
    
    # Get the parameter dictionary
    if x_param == 'delay':
        param_dict = results['delay_dict']
        param_label = 'Delay'
        param_unit = ' μs'
    elif x_param == 'voltage':
        param_dict = results['voltage_dict']
        param_label = 'Voltage'
        param_unit = ' V'
    elif x_param == 'eta':
        param_dict = results['eta_dict']
        param_label = 'η'
        param_unit = '°'
    elif x_param == 'run':
        param_dict = {run: run for run in results['diff_stack'].keys()}
        param_label = 'Run'
        param_unit = ''
    else:
        raise ValueError("x_param must be 'delay', 'voltage', 'eta', or 'run'")
    
    # Ensure all parameter values are floats (except run which might be int)
    if x_param != 'run':
        param_dict = {k: float(v) if v is not None else None for k, v in param_dict.items()}
        # Remove None values
        param_dict = {k: v for k, v in param_dict.items() if v is not None}
    
    # Get difference images
    diff_stack = results['diff_stack']
    
    # Get moments for COM markers
    com_g1_dict = results.get('com_g1', {})
    com_g2_dict = results.get('com_g2', {})
    
    # Validate show_com
    if show_com and not roi:
        print("Warning: ROI needed for COM display. Disabling show_com.")
        show_com = False

    if use_absolute_pixels and not roi:
        print("Warning: ROI needed for absolute pixel positioning. Using relative pixels.")
        use_absolute_pixels = False

    # Get ROI offsets for absolute pixel positioning
    if roi is not None:
        roi_row_start, roi_row_end, roi_col_start, roi_col_end = roi
    else:
        roi_row_start = roi_col_start = 0
        
    animation_widgets = []
    
    # Set up key ranges
    if not key_ranges:
        valid_values = [v for v in param_dict.values() if v is not None]
        if valid_values:
            key_ranges = [(min(valid_values), max(valid_values))]
        else:
            print("No valid parameter values found!")
            return []
    
    for start, end in key_ranges:
        # Ensure start and end are floats (if not run mode)
        if x_param != 'run':
            start = float(start)
            end = float(end)
        
        # Filter runs in this parameter range
        subset_runs = [
            run for run, param_val in param_dict.items()
            if param_val is not None and start <= param_val <= end
        ]
        
        if not subset_runs:
            print(f"Skipping range ({start} to {end}): No matching runs found.")
            continue
        
        # Sort by parameter value
        subset_runs.sort(key=lambda run: param_dict[run])
        
        print(f"Creating animation for {x_param} range [{start}, {end}] with {len(subset_runs)} runs...")
        
        # Create figure
        fig, ax = plt.subplots(figsize=figsize)
        
        # Initialize with first frame
        first_run = subset_runs[0]
        first_param = param_dict[first_run]
        first_image = diff_stack[first_run]

         # Set up normalization based on log_scale parameter
        if log_scale:
            # Use SymLogNorm for data that includes negative values
            # This is appropriate for difference images
            norm = colors.SymLogNorm(linthresh=linthresh, vmin=vmin, vmax=vmax, base=10)
            scale_type = f"SymLog (linear ±{linthresh})"
        else:
            # Use linear normalization
            norm = colors.Normalize(vmin=vmin, vmax=vmax)
            scale_type = "Linear"
        
        print(f"  Using {scale_type} color scale")
        
        # Set up extent for absolute pixel positioning
        if use_absolute_pixels:
            num_rows, num_cols = first_image.shape
            extent = [roi_col_start, roi_col_start + num_cols, 
                     roi_row_start + num_rows, roi_row_start]
            im = ax.imshow(first_image, cmap=cmap, 
                          norm=norm,
                          extent=extent, origin='upper')
            pixel_label_h = 'Pixel (absolute horizontal)'
            pixel_label_v = 'Pixel (absolute vertical)'
        else:
            im = ax.imshow(first_image, cmap=cmap, 
                          norm=norm)
            pixel_label_h = 'Pixel (relative horizontal)'
            pixel_label_v = 'Pixel (relative vertical)'

         # Add colorbar
        cbar = fig.colorbar(im, ax=ax, orientation='horizontal', pad=0.05)
        cbar_label = 'Intensity Difference (G2 - G1)'
        if log_scale:
            cbar_label += f' [{scale_type}]'
        cbar.set_label(cbar_label, fontsize=10)
        
        # Initialize COM markers if needed
        com1_marker = None
        com2_marker = None
        if show_com:
            com1_marker, = ax.plot([], [], 'go', markersize=8, 
                                  label='G1 COM', markeredgecolor='white', 
                                  markeredgewidth=1.5)
            com2_marker, = ax.plot([], [], 'mo', markersize=8, 
                                  label='G2 COM', markeredgecolor='white', 
                                  markeredgewidth=1.5)
            ax.legend(loc='upper right', fontsize=10)
        
        # Set initial title
        if x_param == 'run':
            title_text = f"Run {first_run}"
        else:
            title_text = f"Run {first_run}, {param_label}: {first_param:.2f}{param_unit}"
        title = ax.set_title(title_text, fontsize=14)
        
        ax.set_xlabel('Pixel (horizontal)', fontsize=12)
        ax.set_ylabel('Pixel (vertical)', fontsize=12)

        # Set axis limits if specified
        if xlim is not None:
            ax.set_xlim(xlim)
        if ylim is not None:
            ax.set_ylim(ylim)
        
        plt.tight_layout()
        
        # Update function for animation
        def update(frame_idx):
            run = subset_runs[frame_idx]
            diff_image = diff_stack[run]
            param_value = param_dict[run]
            
            # Update image
            im.set_data(diff_image)
            
            # Update title
            if x_param == 'run':
                title.set_text(f"Run {run}")
            else:
                title.set_text(f"Run {run}, {param_label}: {param_value:.2f}{param_unit}")
            
            artists = [im, title]
            
            # Update COM markers if shown
            if show_com and com1_marker is not None and run in com_g1_dict and run in com_g2_dict:
                # Get absolute COM
                com1_abs = com_g1_dict[run]
                com2_abs = com_g2_dict[run]
                
                if use_absolute_pixels:
                    # Use absolute pixel positions (COM is already in absolute coordinates)
                    com1_col = com1_abs[1]
                    com1_row = com1_abs[0]
                    com2_col = com2_abs[1]
                    com2_row = com2_abs[0]
                else:
                    # Convert to relative position within ROI
                    com1_row = com1_abs[0] - roi_row_start
                    com1_col = com1_abs[1] - roi_col_start
                    com2_row = com2_abs[0] - roi_row_start
                    com2_col = com2_abs[1] - roi_col_start
                
                com1_marker.set_data([com1_col], [com1_row])
                com2_marker.set_data([com2_col], [com2_row])
                
                artists.extend([com1_marker, com2_marker])
            
            return artists
        
        # Create animation
        ani = FuncAnimation(fig, update, frames=len(subset_runs), 
                           interval=interval, blit=True)
        
        # Save if requested
        if save_path:
            path = Path(save_path)
            start_str = str(start).replace('.', '_')
            end_str = str(end).replace('.', '_')
            
            if save_html:
                # Save as interactive HTML with slider
                subset_path = path.parent / f"{path.stem}_{x_param}_{start_str}_to_{end_str}.html"
                print(f"Saving interactive HTML for {x_param} range [{start}, {end}] to {subset_path.name}...")
                
                html_content = ani.to_jshtml()
                with open(subset_path, 'w') as f:
                    f.write(html_content)
                
                print(f"Saved! Open {subset_path.name} in a web browser to view with slider.")
            else:
                # Save as video (.gif or .mp4)
                subset_path = path.parent / f"{path.stem}_{x_param}_{start_str}_to_{end_str}{path.suffix}"
                fps = int(1000 / interval)
                file_extension = subset_path.suffix.lower()
                
                if file_extension == '.gif':
                    print(f"Saving {x_param} range [{start}, {end}] to {subset_path.name}...")
                    ani.save(subset_path, writer='pillow', fps=fps)
                elif file_extension == '.mp4':
                    print(f"Saving {x_param} range [{start}, {end}] to {subset_path.name}...")
                    ani.save(subset_path, writer='ffmpeg', fps=fps, bitrate=1800)
                else:
                    print(f"Warning: Extension '{file_extension}' not recognized. Save skipped.")
        
        plt.close(fig)
        
        # Display animation if requested
        html_player = HTML(ani.to_jshtml())
        if display_animation:
            print(f"\n=== Animation {param_label} Range: {start} to {end} ===")
            display(html_player)
        
        animation_widgets.append(html_player)
    
    return animation_widgets


def animate_lineouts(results, x_param='delay', roi=None, 
                     key_ranges=None, ylim=None, ylim_diff=None, ylim_fra_diff=None,
                     xlim=None,
                     show_difference=True, show_com=True,
                     use_absolute_pixels=True,
                     interval=200, save_path=None, save_html=False, figsize=(18, 5)):
    """
    Animate lineouts as a function of voltage or delay time.
    
    Parameters:
    -----------
    results : dict
        Output from process_scan_data containing moments and lineout data
    x_param : str, optional
        Parameter to animate over: 'delay' or 'voltage' (default: 'delay')
    roi : list, optional
        ROI used for data extraction [row_start, row_end, col_start, col_end]
    key_ranges : list of tuples, optional
        List of (min, max) bounds for x_param to create separate animations
        Example: [(10, 50), (100, 150)]
    ylim : tuple, optional
        Y-axis limits for lineout plots (default: auto)
    ylim_diff : tuple, optional
        Y-axis limits for difference plot (default: auto)
    ylim_fra_diff : tuple, optional
        Y-axis limits for fractional difference plot (default: auto)
    xlim : tuple, optional
        X-axis limits for lineout plot (default: auto)
    show_difference : bool, optional
        Whether to show G2-G1 difference plot (default: True)
    show_com : bool, optional
        Whether to show COM vertical lines (default: True)
    use_absolute_pixels : bool, optional
        If True, uses absolute pixel positions from detector (default: True)
    interval : int, optional
        Time between frames in milliseconds (default: 200)
    save_path : str, optional
        Base filename for saving animations (.gif or .mp4)
    save_html : bool, optional
        If True, saves interactive HTML with slider instead of video (default: False)
    figsize : tuple, optional
        Figure size (default: (18, 5))
    
    Returns:
    --------
    animation_widgets : list
        List of HTML animation widgets
    """
    from matplotlib.animation import FuncAnimation
    from IPython.display import HTML, display
    from pathlib import Path
    import matplotlib.pyplot as plt
    import numpy as np
    
    # Get the parameter dictionary
    if x_param == 'delay':
        param_dict = results['delay_dict']
        param_label = 'Delay (μs)'
    elif x_param == 'voltage':
        param_dict = results['voltage_dict']
        param_label = 'Voltage (V)'
    elif x_param == 'eta':
        param_dict = results['eta_dict']
        param_label = 'eta (deg)'
    else:
        raise ValueError("x_param must be 'delay', 'voltage', or 'eta'")
    
    # Ensure all parameter values are floats
    param_dict = {k: float(v) for k, v in param_dict.items()}
    
    # Get lineout data
    lineout_1_dict = results['lineout_1_dict']
    lineout_2_dict = results['lineout_2_dict']
    
    # Get moments for COM markers
    com_g1_dict = results.get('com_g1', {})
    com_g2_dict = results.get('com_g2', {})
    
    # Get ROI offset for absolute pixel positions
    if roi is None:
        roi_col_start = 0
    else:
        roi_col_start = roi[2]
    
    animation_widgets = []
    
    if not key_ranges:
        key_ranges = [(min(param_dict.values()), max(param_dict.values()))]
    
    for start, end in key_ranges:
        # Ensure start and end are floats
        start = float(start)
        end = float(end)
        
        # Filter runs in this parameter range
        subset_runs = [
            run for run, param_val in param_dict.items()
            if start <= float(param_val) <= end
        ]
        
        if not subset_runs:
            print(f"Skipping range ({start} to {end}): No matching runs found.")
            continue
        
        # Sort by parameter value
        subset_runs.sort(key=lambda run: float(param_dict[run]))
        
        print(f"Creating animation for {x_param} range [{start}, {end}] with {len(subset_runs)} runs...")
        
        # Initialize figure
        if show_difference:
            fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=figsize)
        else:
            fig, ax1 = plt.subplots(1, 1, figsize=(figsize[0]//3, figsize[1]))
        
        # Get first run's data to initialize
        first_run = subset_runs[0]
        first_param = float(param_dict[first_run])
        
        # Create pixel positions (absolute or relative)
        num_pixels = len(lineout_1_dict[first_run])
        if use_absolute_pixels:
            pixels = np.arange(roi_col_start, roi_col_start + num_pixels)
            pixel_label = 'Pixel (absolute)'
        else:
            pixels = np.arange(num_pixels)
            pixel_label = 'Pixel (relative)'
        
        # Initial plots - Axis 1: Lineouts
        line1_g1, = ax1.plot(pixels, lineout_1_dict[first_run], label='Gate 1', color='tab:blue')
        line1_g2, = ax1.plot(pixels, lineout_2_dict[first_run], label='Gate 2', color='tab:orange')
        
        # Add COM markers if available
        vline1_g1 = None
        vline1_g2 = None
        if show_com and first_run in com_g1_dict and first_run in com_g2_dict:
            com_g1_col = com_g1_dict[first_run][1]
            com_g2_col = com_g2_dict[first_run][1]
            
            vline1_g1 = ax1.axvline(x=com_g1_col, color='tab:blue', 
                                    linestyle='--', linewidth=1, alpha=0.7, label='G1 COM')
            vline1_g2 = ax1.axvline(x=com_g2_col, color='tab:orange', 
                                    linestyle='--', linewidth=1, alpha=0.7, label='G2 COM')
        
        ax1.set_xlabel(pixel_label, fontsize=14)
        ax1.set_ylabel('Summed Intensity', fontsize=14)
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        title1 = ax1.set_title(f'{param_label}: {first_param:.2f}', fontsize = 15)
        
        if ylim is not None:
            ax1.set_ylim(ylim)
        if xlim is not None:
            ax1.set_xlim(xlim)
        
        # Initialize variables for axes 2 and 3
        line2_diff = None
        title2 = None
        line3_fra_diff = None
        title3 = None
        
        if show_difference:
            # Axis 2: Absolute Difference plot
            diff_data = lineout_2_dict[first_run] - lineout_1_dict[first_run]
            line2_diff, = ax2.plot(pixels, diff_data, 'o-', color='tab:red', markersize=3)
            ax2.set_xlabel(pixel_label, fontsize=14)
            ax2.set_ylabel('Intensity Difference (G2-G1)', fontsize=14)
            ax2.grid(True, alpha=0.3)
            title2 = ax2.set_title(f'{param_label}: {first_param:.2f}', fontsize = 15)
            
            if ylim_diff is not None:
                ax2.set_ylim(ylim_diff)
            if xlim is not None:
                ax2.set_xlim(xlim)
            
            # Axis 3: Fractional Difference plot
            fra_diff_data = (lineout_2_dict[first_run] - lineout_1_dict[first_run]) / lineout_1_dict[first_run]
            line3_fra_diff, = ax3.plot(pixels, fra_diff_data, 'o-', color='tab:purple', markersize=3)
            ax3.set_xlabel(pixel_label, fontsize=14)
            ax3.set_ylabel('Frac. Diff. (G2-G1)/G1', fontsize=14)
            ax3.grid(True, alpha=0.3)
            title3 = ax3.set_title(f'{param_label}: {first_param:.2f}', fontsize = 15)
            
            if ylim_fra_diff is not None:
                ax3.set_ylim(ylim_fra_diff)
            if xlim is not None:
                ax3.set_xlim(xlim)
        
        plt.tight_layout()
        
        # Animation update function
        def update(frame_index):
            run = subset_runs[frame_index]
            param_val = float(param_dict[run])
            
            # Update lineouts
            line1_g1.set_ydata(lineout_1_dict[run])
            line1_g2.set_ydata(lineout_2_dict[run])
            
            # Update COM markers
            if show_com and vline1_g1 is not None and run in com_g1_dict:
                com_g1_col = com_g1_dict[run][1]
                vline1_g1.set_xdata([com_g1_col, com_g1_col])
            if show_com and vline1_g2 is not None and run in com_g2_dict:
                com_g2_col = com_g2_dict[run][1]
                vline1_g2.set_xdata([com_g2_col, com_g2_col])
            
            # Update title
            title1.set_text(f'{param_label}: {param_val:.2f}')
            
            # Build return list
            artists = [line1_g1, line1_g2, title1]
            if vline1_g1:
                artists.extend([vline1_g1, vline1_g2])
            
            # Update absolute difference plot
            if show_difference and line2_diff is not None:
                diff_data = lineout_2_dict[run] - lineout_1_dict[run]
                line2_diff.set_ydata(diff_data)
                title2.set_text(f'{param_label}: {param_val:.2f}')
                artists.extend([line2_diff, title2])
            
            # Update fractional difference plot
            if show_difference and line3_fra_diff is not None:
                fra_diff_data = (lineout_2_dict[run] - lineout_1_dict[run]) / lineout_1_dict[run]
                line3_fra_diff.set_ydata(fra_diff_data)
                title3.set_text(f'{param_label}: {param_val:.2f}')
                artists.extend([line3_fra_diff, title3])
            
            return artists
        
        # Create animation
        ani = FuncAnimation(
            fig, update, frames=len(subset_runs), interval=interval, blit=True
        )
        
        # Save if requested
        if save_path:
            path = Path(save_path)
            start_str = str(start).replace('.', '_')
            end_str = str(end).replace('.', '_')
            
            if save_html:
                # Save as interactive HTML with slider
                subset_path = path.parent / f"{path.stem}_{x_param}_{start_str}_to_{end_str}.html"
                print(f"Saving interactive HTML for {x_param} range [{start}, {end}] to {subset_path.name}...")
                
                html_content = ani.to_jshtml()
                with open(subset_path, 'w') as f:
                    f.write(html_content)
                
                print(f"Saved! Open {subset_path.name} in a web browser to view with slider.")
            else:
                # Save as video (.gif or .mp4)
                subset_path = path.parent / f"{path.stem}_{x_param}_{start_str}_to_{end_str}{path.suffix}"
                fps = int(1000 / interval)
                file_extension = subset_path.suffix.lower()
                
                if file_extension == '.gif':
                    print(f"Saving {x_param} range [{start}, {end}] to {subset_path.name}...")
                    ani.save(subset_path, writer='pillow', fps=fps)
                elif file_extension == '.mp4':
                    print(f"Saving {x_param} range [{start}, {end}] to {subset_path.name}...")
                    ani.save(subset_path, writer='ffmpeg', fps=fps, bitrate=1800)
        
        plt.close(fig)
        
        # Display animation
        html_player = HTML(ani.to_jshtml())
        print(f"\n=== Animation {param_label} Range: {start} to {end} ===")
        display(html_player)
        
        animation_widgets.append(html_player)
    
    return animation_widgets

In [ ]:
#Defining functions to use throughout the notebook.
def calc_moments(img, roi, background_roi=None):
    """
     Calculate image moments including COM, variance, skewness, and kurtosis.
    
    Parameters:
    -----------
    img : ndarray
        2D image array
    roi : tuple, optional
        Region of interest as (row_start, row_end, col_start, col_end)
    
    Returns:
    --------
    moments : dict
        Dictionary containing calculated moments
    """
        
    # Extract signal region, this is the entire ROI
    img_signal = img[roi[0]:roi[1], roi[2]:roi[3]]
    
    # Calculate background
    if background_roi is not None:
        background_region = img[background_roi[0]:background_roi[1], background_roi[2]:background_roi[3]]
        background = np.nanmean(background_region)
    else:
        background = 0
    
    # Subtract background
    img_clean = np.nan_to_num(img_signal, nan=0) - background
    
    # Create coordinate grids for signal ROI
    rows_grid, col_grid = np.mgrid[roi[0]:roi[1], roi[2]:roi[3]]
    img_flat = img_clean.flatten()
    rows_flat = rows_grid.flatten()
    cols_flat = col_grid.flatten()
    total_intensity = np.sum(img_flat)
    
    if total_intensity <= 0:
        return {'total_intensity': 0, 'com': [np.nan, np.nan], 'com_rel': [np.nan, np.nan], 'variance': [np.nan, np.nan], 'skewness': [np.nan, np.nan],
            'kurtosis': [np.nan, np.nan],'background': background}
    
    com_row = np.sum(img_flat * rows_flat) / total_intensity
    com_col = np.sum(img_flat * cols_flat) / total_intensity
    var_row = np.sum(img_flat * (rows_flat - com_row)**2) / total_intensity
    var_col = np.sum(img_flat * (cols_flat - com_col)**2) / total_intensity
    skew_row = (np.sum(img_flat * (rows_flat - com_row)**3) / (total_intensity * var_row**1.5))
    skew_col = (np.sum(img_flat * (cols_flat - com_col)**3) / (total_intensity * var_col**1.5))
    kurt_row = (np.sum(img_flat * (rows_flat - com_row)**4) / (total_intensity * var_row**2))
    kurt_col = (np.sum(img_flat * (cols_flat - com_col)**4) / (total_intensity * var_col**2))
    
    #com_rel = [com_row - roi[0], com_col - roi[2]]

    moments = {
        'background': background,
        'tot_int': np.array([total_intensity]),
        'com_rel': np.array([com_row - roi[0], com_col - roi[2]]),  # Relative to ROI
        'com_abs': np.array([com_row , com_col]),  # Absolute position
        'variance': np.array([var_row, var_col]),
        'skewness': np.array([skew_row, skew_col]),
        'kurtosis': np.array([kurt_row, kurt_col])
    }
    
    return moments

def print_moments(moments_g1, moments_g2=None, keys=None, run=None, precision=2, labels=None, show_diff=True):
    """
    Print selected moments from calc_moments output.
    
    Parameters:
    -----------
    moments_g1 : dict
        Moments dictionary from calc_moments for Gate 1
    moments_g2 : dict, optional
        Moments dictionary from calc_moments for Gate 2
    keys : list of str, optional
        List of keys to print from moments dict. 
        If None, prints all available keys.
        Available: 'background', 'total_intensity', 'com', 'com_rel', 
                   'variance', 'skewness', 'kurtosis'
    run : int or str, optional
        Run number to include in header
    precision : int, optional
        Number of decimal places for floating point values (default: 2)
    labels : dict, optional
        Custom labels for gates, e.g., {'g1': 'Gate 1', 'g2': 'Gate 2'}
    show_diff : bool, optional
        Whether to show G2-G1 difference (default: True)
    """
    # Default labels
    if labels is None:
        labels = {'g1': 'G1', 'g2': 'G2'}
    
    # If no keys specified, use all available keys
    if keys is None:
        keys = list(moments_g1.keys())
    
    # Print header
    if run is not None:
        print(f"Run {run}:")
    
    # Print each key
    for key in keys:
        # Format the key name for display
        key_display = key.replace('_', ' ').title()
        
        # Print G1 value
        if key in moments_g1:
            value_g1 = moments_g1[key]
            if isinstance(value_g1, (list, np.ndarray)):
                formatted_g1 = f"[{', '.join([f'{v:.{precision}f}' for v in value_g1])}]"
            elif isinstance(value_g1, (int, float, np.number)):
                formatted_g1 = f"{value_g1:.{precision}f}"
            else:
                formatted_g1 = str(value_g1)
            
            print(f"  {labels['g1']} {key_display}: {formatted_g1}")
        
        # Print G2 value if provided
        if moments_g2 is not None and key in moments_g2:
            value_g2 = moments_g2[key]
            if isinstance(value_g2, (list, np.ndarray)):
                formatted_g2 = f"[{', '.join([f'{v:.{precision}f}' for v in value_g2])}]"
            elif isinstance(value_g2, (int, float, np.number)):
                formatted_g2 = f"{value_g2:.{precision}f}"
            else:
                formatted_g2 = str(value_g2)
            
            print(f"  {labels['g2']} {key_display}: {formatted_g2}")

            # Print difference (G2 - G1) if requested
            if show_diff and key in moments_g1:
                try:
                    if isinstance(value_g2, (list, np.ndarray)):
                        diff = np.array(value_g2) - np.array(value_g1)
                        formatted_diff = f"[{', '.join([f'{v:.{precision}f}' for v in diff])}]"
                    elif isinstance(value_g2, (int, float, np.number)):
                        diff = value_g2 - value_g1
                        formatted_diff = f"{diff:.{precision}f}"
                    else:
                        formatted_diff = None
                    
                    if formatted_diff is not None:
                        print(f"  {labels['g2']} - {labels['g1']} {key_display}: {formatted_diff}")
                except (TypeError, ValueError):
                    # Skip difference if subtraction not possible
                    pass

def visualize_rois(data, signal_roi, background_roi):
    """Show signal and background ROIs on image"""
    from matplotlib.patches import Rectangle
    
    fig, ax = plt.subplots(figsize=(12, 10))
    ax.imshow(data, norm=colors.LogNorm(vmin=1, vmax=1e4))
    
    # Signal ROI (red)
    signal_rect = Rectangle(
        (signal_roi[2], signal_roi[0]), 
        signal_roi[3]-signal_roi[2], 
        signal_roi[1]-signal_roi[0],
        linewidth=2, edgecolor='r', facecolor='none', label='Signal ROI'
    )
    ax.add_patch(signal_rect)
    
    # Background ROI (green)
    bg_rect = Rectangle(
        (background_roi[2], background_roi[0]), 
        background_roi[3]-background_roi[2], 
        background_roi[1]-background_roi[0],
        linewidth=2, edgecolor='g', facecolor='none', label='Background ROI'
    )
    ax.add_patch(bg_rect)
    
    ax.legend()
    ax.set_title('ROI Selection')
    plt.show()

def remove_hot_pixels(data, simple_threshold=4e5, check_spacing=1):
    """
    Remove hot pixels by checking sparse grid of pixels
    Much faster for large images
    
    Parameters:
    -----------
    data : 2D array
    simple_threshold : float
        Absolute threshold for hot pixels
    check_spacing : int
        Check every Nth pixel (e.g., 5 = check every 5th pixel)
    """
    from scipy.ndimage import median_filter
    
    data_clean = data.copy().astype(float)
    
    # Method 1: Simple absolute threshold (applied to all pixels)
    data_clean = np.where(data_clean > simple_threshold, np.nan, data_clean)
    
    # Method 2: Sparse neighborhood check
    # Only check pixels on a sparse grid
    rows, cols = data_clean.shape
    
    # Create median filtered version (3x3 neighborhood)
    median_filtered = median_filter(np.nan_to_num(data_clean, nan=0), size=3)
    
    # Only check every Nth pixel
    for i in range(0, rows, check_spacing):
        for j in range(0, cols, check_spacing):
            pixel_val = data_clean[i, j]
            median_val = median_filtered[i, j]
            
            # If pixel is much brighter than its neighborhood median
            if not np.isnan(pixel_val) and median_val > 0:
                if pixel_val > 10 * median_val:  # 10x threshold
                    data_clean[i, j] = np.nan
    
    return data_clean


def parse_filename(file, fields=['run', 'voltage', 'delay', 'eta']):
    """
    Parse specified fields from filename
    
    Parameters:
    -----------
    file : Path object
    fields : list of str
        Which fields to parse: 'run', 'voltage', 'delay', 'eta'
    
    Returns:
    --------
    dict with requested fields (only includes fields found in filename)
    """
    import re
    result = {}
    filename = file.stem
    
    #print(f"DEBUG: Parsing filename: {filename}")  # DEBUG
    
    if 'run' in fields:
        match = re.search(r'run(\d+)', filename)
        if match:
            result['run'] = int(match.group(1))
            #print(f"DEBUG: Found run: {result['run']}")  # DEBUG
    
    if 'voltage' in fields:
        match = re.search(r'(\d+(?:p\d+)?m?V)', filename)
        if match:
            voltage = match.group(1).replace('V', '')
            voltage = voltage.replace('m', 'e-3').replace('p', '.')
            result['voltage'] = float(voltage) * 20
            #print(f"DEBUG: Found voltage: {result['voltage']}")  # DEBUG
    
    if 'delay' in fields:
        match = re.search(r'delay([\d.p]+)us', filename)
        #print(f"DEBUG: delay match object: {match}")  # DEBUG
        if match:
            result['delay'] = match.group(1).replace('p', '.')
            #print(f"DEBUG: Found delay: {result['delay']}")  # DEBUG
        else:
            print(f"DEBUG: delay NOT found in filename")  # DEBUG
        
    if 'eta' in fields:
        match = re.search(r'eta([\d.p]+)', filename)
        if match:
            result['eta'] = match.group(1).replace('p', '.')
            #print(f"DEBUG: Found eta: {result['eta']}")  # DEBUG
    
    return result

def plot_lineouts(lineout_1, lineout_2, moments_g1=None, moments_g2=None, run=None, voltage=None, delay=None, eta = None, 
                  figsize=(18, 4), show_com=True):
    """
    Plot two lineouts and their difference.
    
    Parameters:
    -----------
    lineout_1 : array-like
        First lineout data (Gate 1)
    lineout_2 : array-like
        Second lineout data (Gate 2)
    moments_g1 : dict, optional
        Dictionary containing 'com_rel' key for Gate 1 center of mass
    moments_g2 : dict, optional
        Dictionary containing 'com_rel' key for Gate 2 center of mass
    figsize : tuple, optional
        Figure size (default: (12, 4))
    show_com : bool, optional
        Whether to show COM vertical lines (default: True)
    
    Returns:
    --------
    fig, ax : matplotlib figure and axes objects
    """
    import matplotlib.pyplot as plt
    import numpy as np
    
    fig, ax = plt.subplots(1, 3, figsize=figsize)
    
    # Left panel: both lineouts
    ax[0].plot(lineout_1, label='Gate 1')
    ax[0].plot(lineout_2, label='Gate 2')
    
    # Add vertical lines for centers of mass if requested
    if show_com and moments_g1 is not None and moments_g2 is not None:
        ax[0].axvline(x=moments_g1['com_rel'][1], color='tab:blue', 
                      linestyle='--', linewidth=1, label='G1 COM')
        ax[0].axvline(x=moments_g2['com_rel'][1], color='tab:orange', 
                      linestyle='--', linewidth=1, label='G2 COM')
    
    ax[0].legend()
    ax[0].set_xlabel('Pixel', fontsize=14)
    ax[0].set_ylabel('Summed Int', fontsize=14)
    ax[0].tick_params(axis='both', labelsize=14, length=6, width=2)
    
    # Middle panel: difference
    ax[1].plot(lineout_2 - lineout_1, label='On-Off')
    ax[1].set_xlabel('Pixel', fontsize=14)
    ax[1].set_ylabel('Summed Intensity Difference (G2-G1)', fontsize=14)
    ax[1].tick_params(axis='both', labelsize=14, length=8, width=2)

    # Right pannel: normalized difference
    ax[2].plot(((lineout_2 - lineout_1)/lineout_1))
    ax[2].set_xlabel('Pixel', fontsize=14)
    ax[2].set_ylabel('ΔI/I (G2-G1)/(G1)', fontsize=14)
    ax[2].tick_params(axis='both', labelsize=14, length=8, width=2)

    # Add title if parameters provided
    if run is not None or voltage is not None or delay is not None or eta is not None:
        title_parts = []
        if run is not None:
            title_parts.append(f'run: {run}')
        if voltage is not None:
            title_parts.append(f'Voltage: {voltage:.2f} V')
        if delay is not None:
            title_parts.append(f'Delay: {delay} μs')
        if eta is not None:
            title_parts.append(f'Eta: {eta}°')
        fig.suptitle(', '.join(title_parts))
    
    return fig, ax

def plot_diff_image(data, roi, moments_g1, moments_g2, run=None, voltage=None, delay=None, eta = None,
                    vmin=-300, vmax=300, cmap='bwr', figsize=None, show_com=True):
    """
    Plot difference image between two gates with centers of mass.
    
    Parameters:
    -----------
    data : array-like
        Data array where data[0] is Gate 1 and data[1] is Gate 2
    roi : tuple or list
        Region of interest as (row_start, row_end, col_start, col_end)
    moments_g1 : dict
        Dictionary containing 'com_rel' key for Gate 1 center of mass
    moments_g2 : dict
        Dictionary containing 'com_rel' key for Gate 2 center of mass
    run : int or str, optional
        Run number for title
    voltage : float, optional
        Voltage value for title
    delay : float, optional
        Delay value for title
    vmin : float, optional
        Minimum value for colormap normalization (default: -300)
    vmax : float, optional
        Maximum value for colormap normalization (default: 300)
    cmap : str, optional
        Colormap name (default: 'bwr')
    figsize : tuple, optional
        Figure size (default: None, uses matplotlib default)
    show_com : bool, optional
        Whether to show COM markers (default: True)
    
    Returns:
    --------
    fig, ax : matplotlib figure and axes objects
    """
    import matplotlib.pyplot as plt
    import matplotlib.colors as colors
    
    fig, ax = plt.subplots(figsize=figsize)
    
    # Calculate difference image
    diff_image = data[1][roi[0]:roi[1], roi[2]:roi[3]] - data[0][roi[0]:roi[1], roi[2]:roi[3]]
    
    # Display difference image
    im = ax.imshow(diff_image, norm=colors.Normalize(vmin=vmin, vmax=vmax), cmap=cmap)
    
    # Plot COMs if requested
    if show_com:
        ax.plot(moments_g1['com_rel'][1], moments_g1['com_rel'][0], 'go', 
                markersize=5, label='G1 COM', markeredgecolor='white', markeredgewidth=1)
        ax.plot(moments_g2['com_rel'][1], moments_g2['com_rel'][0], 'mo', 
                markersize=5, label='G2 COM', markeredgecolor='white', markeredgewidth=1)
        ax.legend()
    
    fig.colorbar(im, ax=ax, orientation='horizontal')
    
    # Add title if parameters provided
    if run is not None or voltage is not None or delay is not None or eta is not None:
        title_parts = []
        if run is not None:
            title_parts.append(f'run: {run}')
        if voltage is not None:
            title_parts.append(f'Voltage: {voltage:.2f} V')
        if delay is not None:
            title_parts.append(f'Delay: {delay} μs')
        if eta is not None:
            title_parts.append(f'Eta: {eta}°')
        fig.suptitle(', '.join(title_parts))
    
    return fig, ax

def plot_comparison(x, y1, y2, xlabel='X', ylabel='Y', 
                   label1='Gate 1', label2='Gate 2',
                   marker='o', figsize=(28,5)):
    """
    Plot two datasets and their difference vs a common x variable.
    
    Parameters:
    -----------
    x : array-like
        X-axis data (e.g., voltage, time, etc.)
    y1 : array-like
        First dataset (e.g., Gate 1 intensity)
    y2 : array-like
        Second dataset (e.g., Gate 2 intensity)
    xlabel : str, optional
        Label for x-axis (default: 'X')
    ylabel : str, optional
        Label for y-axis (default: 'Y')
    label1 : str, optional
        Label for first dataset (default: 'Gate 1')
    label2 : str, optional
        Label for second dataset (default: 'Gate 2')
    marker : str, optional
        Marker style (default: 'o')
    figsize : tuple, optional
        Figure size (default: (15,5))
    
    Returns:
    --------
    fig, ax : matplotlib figure and axes objects
    """
    fig, ax = plt.subplots(1, 3, figsize=figsize)
    
    # Left panel: both datasets
    ax[0].plot(x, y1, marker, label=label1)
    ax[0].plot(x, y2, marker, label=label2)
    ax[0].set_xlabel(xlabel, fontsize=14)
    ax[0].set_ylabel(ylabel, fontsize=14)
    ax[0].tick_params(axis='both', labelsize=14, length=8, width=2)
    ax[0].legend()
    ax[0].grid(True, alpha=0.3)
    
    # Right panel: difference
    ax[1].plot(x, np.array(y2) - np.array(y1), marker)
    ax[1].set_xlabel(xlabel, fontsize=14)
    ax[1].set_ylabel(f'{ylabel} Difference', fontsize=14)
    ax[1].tick_params(axis='both', labelsize=14, length=8, width=2)
    ax[1].legend(['G2-G1'])
    ax[1].grid(True, alpha=0.3)

    # Right pannel: normalized difference
    ax[2].plot(x, ((np.array(y2) - np.array(y1))/np.array(y1)), marker)
    ax[2].set_xlabel(xlabel, fontsize=14)
    ax[2].set_ylabel(f'{ylabel} Fract. Difference', fontsize=14)
    ax[2].tick_params(axis='both', labelsize=14, length=8, width=2)
    ax[2].legend(['(G2-G1)/G1'])
    ax[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    return fig, ax

def plot_comparison_two(x, y1, y2, xlabel='X', ylabel='Y', 
                   label1='Gate 1', label2='Gate 2',
                   marker='o', figsize=(28,5)):
    """
    Plot two datasets and their difference vs a common x variable.
    
    Parameters:
    -----------
    x : array-like
        X-axis data (e.g., voltage, time, etc.)
    y1 : array-like
        First dataset (e.g., Gate 1 intensity)
    y2 : array-like
        Second dataset (e.g., Gate 2 intensity)
    xlabel : str, optional
        Label for x-axis (default: 'X')
    ylabel : str, optional
        Label for y-axis (default: 'Y')
    label1 : str, optional
        Label for first dataset (default: 'Gate 1')
    label2 : str, optional
        Label for second dataset (default: 'Gate 2')
    marker : str, optional
        Marker style (default: 'o')
    figsize : tuple, optional
        Figure size (default: (15,5))
    
    Returns:
    --------
    fig, ax : matplotlib figure and axes objects
    """
    fig, ax = plt.subplots(1, 2, figsize=figsize)
    
    # Left panel: both datasets
    ax[0].plot(x, y1, marker, label=label1)
    ax[0].plot(x, y2, marker, label=label2)
    ax[0].set_xlabel(xlabel, fontsize=14)
    ax[0].set_ylabel(ylabel, fontsize=14)
    ax[0].tick_params(axis='both', labelsize=14, length=8, width=2)
    ax[0].legend()
    ax[0].grid(True, alpha=0.3)
    
    # Right panel: difference
    # ax[1].plot(x, np.array(y2) - np.array(y1), marker)
    # ax[1].set_xlabel(xlabel, fontsize=14)
    # ax[1].set_ylabel(f'{ylabel} Difference', fontsize=14)
    # ax[1].tick_params(axis='both', labelsize=14, length=8, width=2)
    # ax[1].legend(['G2-G1'])
    # ax[1].grid(True, alpha=0.3)

    # Right pannel: normalized difference
    ax[1].plot(x, ((np.array(y2) - np.array(y1))/np.array(y1)), marker)
    ax[1].set_xlabel(xlabel, fontsize=14)
    ax[1].set_ylabel(f'{ylabel} Fract. Difference', fontsize=14)
    ax[1].tick_params(axis='both', labelsize=14, length=8, width=2)
    ax[1].legend(['(G2-G1)/G1'])
    ax[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    return fig, ax

In [ ]:
def plot_static_eta(data_file, eta_range, row_crop=None, simple_threshold=4e5, check_spacing=1,
                    vmin=1e0, vmax=1e4, ylim=None, figsize_image=(10,5), figsize_plot=(6,4)):
    """
    Plot static eta scan data: middle slice image and intensity vs eta.
    
    Parameters:
    -----------
    data_file : str or Path
        Path to the HDF5 data file
    eta_range : tuple
        (eta_min, eta_max) for the eta values
    row_crop : tuple, optional
        (row_start, row_end) for cropping rows. If None, no cropping.
        Default: None
    simple_threshold : float, optional
        Absolute threshold for hot pixels (default: 4e5)
    check_spacing : int, optional
        Check every Nth pixel for hot pixel detection (default: 1)
    vmin : float, optional
        Minimum value for image log normalization (default: 1e0)
    vmax : float, optional
        Maximum value for image log normalization (default: 1e4)
    ylim : tuple, optional
        Y-axis limits for intensity plot, e.g., (2e5, 2e6)
    figsize_image : tuple, optional
        Figure size for image plot (default: (10,5))
    figsize_plot : tuple, optional
        Figure size for intensity plot (default: (6,4))
    
    Returns:
    --------
    etas : ndarray
        Array of eta values
    intensities : ndarray
        Array of integrated intensities
    figs : dict
        Dictionary containing 'image' and 'plot' figure objects
    """
    # Load data
    h5 = h5py.File(data_file, 'r')
    
    if row_crop is not None:
        data = h5['entry/data/data'][:, row_crop[0]:row_crop[1], :]
    else:
        data = h5['entry/data/data'][:]
    
    h5.close()
    
    # Remove hot pixels for each frame
    print("Removing hot pixels...")
    data_clean = np.zeros_like(data, dtype=float)
    for i in range(data.shape[0]):
        data_clean[i] = remove_hot_pixels(data[i], simple_threshold=simple_threshold, check_spacing=check_spacing)
    
    # Create eta array
    num_points = data_clean.shape[0]
    etas = np.linspace(eta_range[0], eta_range[1], num_points)
    
    # Plot middle slice
    fig_image, ax_image = plt.subplots(figsize=figsize_image)
    im = ax_image.imshow(data_clean[data_clean.shape[0]//2], 
                         norm=colors.LogNorm(vmin=vmin, vmax=vmax))
    ax_image.set_title('Middle Slice')
    fig_image.colorbar(im, ax=ax_image, orientation = 'horizontal')
    
    # Calculate intensities
    intensities = np.nansum(data_clean, axis=(1, 2))
    
    # Print normalized intensities
    print("Eta values and normalized intensities:")
    print(*zip(etas, intensities/intensities.max()))
    print(f"Intensities shape: {intensities.shape}")
    
    # Plot intensity vs eta
    fig_plot, ax_plot = plt.subplots(figsize=figsize_plot)
    ax_plot.semilogy(etas, intensities, '.-')
    ax_plot.set_xlabel('Eta')
    ax_plot.set_ylabel('Intensity (sum of pixel values)')
    ax_plot.grid(True)
    
    if ylim is not None:
        ax_plot.set_ylim(ylim)
    
    figs = {'image': fig_image, 'plot': fig_plot}
    
    return etas, intensities, figs

def process_scan_data(h5_loc, h5_pattern='run*data*.h5', 
                     roi=[210, 350, 619, 626],  #003 ctr peak
                     background_roi=[210, 350, 800, 900],
                     exclude_col_range = None, #for cutting out ctr peak
                     run_range=(751, 767),
                     awg_gate_loc = -1,
                     lineout_range=(70, 80),
                     #lineout_axis = 0,
                     plot_diff_image_flag=True,
                     plot_lineouts_flag=True,
                     show_com_on_diff_image=True, 
                     show_com_on_lineouts=True,  
                     diff_image_kwargs=None,
                     lineout_kwargs=None,
                     hot_pixel_threshold=4e5):
    """
    Process scan data: load files, calculate moments, and optionally plot.
    
    Parameters:
    -----------
    h5_loc : Path
        Directory containing HDF5 files
    h5_pattern : str, optional
        Glob pattern for finding HDF5 files (default: 'run*data*.h5')
    roi : list, optional
        Region of interest [row_start, row_end, col_start, col_end]
        Default: [210, 350, 619, 626] (CTR peak only)
    background_roi : list, optional
        Background ROI [row_start, row_end, col_start, col_end]
        Default: [210, 350, 800, 900]
    run_range : tuple, optional
        (min_run, max_run) to process (default: (751, 767))
    lineout_range : tuple, optional
        (start_offset, end_offset) relative to roi[0] for lineout
        Default: (70, 80) means roi[0]+70 to roi[0]+80
    plot_diff_image_flag : bool, optional
        Whether to plot difference images (default: True)
    plot_lineouts_flag : bool, optional
        Whether to plot lineouts (default: True)
    diff_image_kwargs : dict, optional
        Additional kwargs for plot_diff_image (e.g., {'vmin': -500, 'vmax': 500})
    lineout_kwargs : dict, optional
        Additional kwargs for plot_lineouts (e.g., {'figsize': (28, 5)})
    hot_pixel_threshold : float, optional
        Threshold for hot pixel removal (default: 4e5)
    
    Returns:
    --------
    results : dict
        Dictionary containing:
        - 'diff_stack': OrderedDict of difference images
        - 'norm_diff_stack': OrderedDict of normalized difference images
        - 'voltage_dict': OrderedDict of voltages
        - 'delay_dict': OrderedDict of delays
        - 'intensities_g1', 'intensities_g2': OrderedDict of intensities
        - 'com_g1', 'com_g2': OrderedDict of center of mass
        - 'variance_g1', 'variance_g2': OrderedDict of variances
        - 'skewness_g1', 'skewness_g2': OrderedDict of skewness
        - 'kurtosis_g1', 'kurtosis_g2': OrderedDict of kurtosis
        - 'background_g1', 'background_g2': OrderedDict of backgrounds
    """
    from pathlib import Path
    from collections import OrderedDict
    import h5py
    import numpy as np
    
    h5_loc = Path(h5_loc)
    #h5_files = h5_loc.glob(h5_pattern)
    
    # Initialize storage
    diff_stack = OrderedDict()
    norm_diff_stack = OrderedDict()
    voltage_dict = OrderedDict()
    delay_dict = OrderedDict()
    eta_dict = OrderedDict()
    
    # Moments storage
    intensities_g1 = OrderedDict()
    intensities_g2 = OrderedDict()
    com_g1 = OrderedDict()
    com_g2 = OrderedDict()
    variance_g1 = OrderedDict()
    variance_g2 = OrderedDict()
    skewness_g1 = OrderedDict()
    skewness_g2 = OrderedDict()
    kurtosis_g1 = OrderedDict()
    kurtosis_g2 = OrderedDict()
    background_g1 = OrderedDict()
    background_g2 = OrderedDict()

    # Lineout storage
    lineout_1_dict = OrderedDict()
    lineout_2_dict = OrderedDict()
    
    # Set default kwargs
    if diff_image_kwargs is None:
        diff_image_kwargs = {'vmin': -500, 'vmax': 500}
    if lineout_kwargs is None:
        lineout_kwargs = {'figsize': (28, 5)}

    if run_range is None:
        raise ValueError("run_range must be specified as a tuple (min_run, max_run) or list of run numbers")

    # Build file list based on run_range
    h5_files = []

    if isinstance(run_range, tuple):
        # Original behavior: range of runs
        for run in range(run_range[0], run_range[1] + 1):
            h5_files.extend(h5_loc.glob(f'run{run}_*_data_*.h5'))
    elif isinstance(run_range, list):
        # New behavior: specific list of runs
        for run in run_range:
            h5_files.extend(h5_loc.glob(f'run{run}_*_data_*.h5'))
    else:
        raise ValueError("run_range must be either a tuple (min, max) or list of run numbers")
    
    print(f"Found {len(h5_files)} files to process")
    
    # for run in range(run_range[0], run_range[1] + 1):
    #     h5_files.extend(h5_loc.glob(f'run{run}_*_data_*.h5'))
    
    # print(f"Found {len(h5_files)} files to process")
    
    # Process files
    for file in h5_files:
        info = parse_filename(file, fields=['run', 'voltage', 'delay', 'eta'])
        run = info.get('run')

        # Skip if run not found
        if run is None:
            print(f"Warning: Could not parse run number from {file.name}, skipping...")
            continue
            
        voltage = info.get('voltage')  # Fixed: parentheses not brackets
        delay = info.get('delay')      # Fixed: parentheses not brackets
        eta = info.get('eta')          # Fixed: parentheses not brackets
        
        if voltage is not None:
            voltage = -voltage * awg_gate_loc
        
        print(f"Processing run {run}...")
        
        # Load and clean data
        h5 = h5py.File(file, 'r')
        data_raw = h5['entry/data/data'][:]
        h5.close()
        
        # Hot pixel removal
        data = np.zeros_like(data_raw, dtype=float)
        for i in range(data_raw.shape[0]):
            data[i] = remove_hot_pixels(data_raw[i], 
                                       simple_threshold=hot_pixel_threshold)

         # Apply exclusion mask if specified
        if exclude_col_range is not None:
            # Calculate relative exclusion coordinates
            exclude_start_rel = exclude_col_range[0] - roi[2]
            exclude_end_rel = exclude_col_range[1] - roi[2]
            
            # Create masked copy for ROI operations
            data_masked = data.copy()
            data_masked[:, roi[0]:roi[1], roi[2] + exclude_start_rel:roi[2] + exclude_end_rel] = np.nan
            print(f"ROI: {roi[0]}:{roi[1]}, {roi[2] + exclude_start_rel}:{roi[2] + exclude_end_rel}")
        else:
            data_masked = data
        
        # Extract ROI data
        data_1 = data_masked[0, roi[0]:roi[1], roi[2]:roi[3]]
        data_2 = data_masked[1, roi[0]:roi[1], roi[2]:roi[3]]

        # Store lineouts (nansum ignores NaN values)
        lineout_1 = np.nansum(data_masked[0][roi[0]+lineout_range[0]:roi[0]+lineout_range[1], roi[2]:roi[3]], axis=0)
        lineout_2 = np.nansum(data_masked[1][roi[0]+lineout_range[0]:roi[0]+lineout_range[1], roi[2]:roi[3]], axis=0)

        # lineout_1 = np.nanmean(data_masked[0][roi[0]:roi[1], roi[2]:roi[3]], axis=0)
        # lineout_2 = np.nanmean(data_masked[1][roi[0]:roi[1], roi[2]:roi[3]], axis=0)
        
        lineout_1_dict[run] = lineout_1
        lineout_2_dict[run] = lineout_2
        
        # Store difference stacks
        diff_stack[run] = data_2 - data_1
        norm_diff_stack[run] = (data_2 - data_1) / data_1
        
        # Store voltage, delay, and eta
        voltage_dict[run] = voltage
        delay_dict[run] = delay
        eta_dict[run] = eta
        
        # Calculate moments (using masked data)
        moments_g1 = calc_moments(data_masked[0], roi, background_roi)
        moments_g2 = calc_moments(data_masked[1], roi, background_roi)
        
        # Store moments in OrderedDicts
        intensities_g1[run] = moments_g1['tot_int']
        intensities_g2[run] = moments_g2['tot_int']
        com_g1[run] = moments_g1['com_abs']
        com_g2[run] = moments_g2['com_abs']
        variance_g1[run] = moments_g1['variance']
        variance_g2[run] = moments_g2['variance']
        skewness_g1[run] = moments_g1['skewness']
        skewness_g2[run] = moments_g2['skewness']
        kurtosis_g1[run] = moments_g1['kurtosis']
        kurtosis_g2[run] = moments_g2['kurtosis']
        background_g1[run] = moments_g1['background']
        background_g2[run] = moments_g2['background']

        # Plot difference image if requested
        if plot_diff_image_flag:
            # Pass show_com parameter to plot_diff_image
            fig, ax = plot_diff_image(data, roi, moments_g1, moments_g2,
                                     run=run, voltage=voltage, delay=delay, eta = eta,
                                     show_com=show_com_on_diff_image,
                                     **diff_image_kwargs)
        
        # Plot lineouts if requested
        if plot_lineouts_flag:
            # Pass show_com parameter to plot_lineouts
            fig, ax = plot_lineouts(lineout_1, lineout_2, moments_g1, moments_g2, run=run, voltage=voltage, delay=delay, eta = eta,
                                   show_com=show_com_on_lineouts,
                                   **lineout_kwargs)
            
    # Package results
    results = {
        'lineout_1_dict': lineout_1_dict,
        'lineout_2_dict': lineout_2_dict,
        'diff_stack': diff_stack,
        'norm_diff_stack': norm_diff_stack,
        'voltage_dict': voltage_dict,
        'delay_dict': delay_dict,
        'eta_dict': eta_dict,
        'intensities_g1': intensities_g1,
        'intensities_g2': intensities_g2,
        'com_g1': com_g1,
        'com_g2': com_g2,
        'variance_g1': variance_g1,
        'variance_g2': variance_g2,
        'skewness_g1': skewness_g1,
        'skewness_g2': skewness_g2,
        'kurtosis_g1': kurtosis_g1,
        'kurtosis_g2': kurtosis_g2,
        'background_g1': background_g1,
        'background_g2': background_g2,
    }
    
    print(f"\nProcessed {len(voltage_dict)} runs successfully!")
    
    return results

def plot_waveform_with_moments(results, moment_key,
                               waveform_path, scope_runs, gate_threshold=2.3,
                               time_crop=(530, 865), waveform_of_run_mV=12, 
                               waveform_of_measurement_mV=5, waveform_scale=20,
                               gate_width_timing_int = 10, 
                               gate_offset=0.35, gate_scale=0.5,
                               xlim=(-25, 200), waveform_ylim=None,
                               delays_to_show=None, figsize=(14, 6), array_index=0,
                               show_gates=True, show_fractional=True):
    """
    Plot moment differences vs delay time with synchronized voltage waveform and gate signals.
    
    Parameters:
    -----------
    results : dict
        Output dictionary from process_scan_data containing all moments and delay_dict
    moment_key : str
        Key for moment to plot from results (e.g., 'intensities', 'com', 'variance')
        Valid keys: 'intensities', 'com', 'variance', 'skewness', 'kurtosis', 'background'
    waveform_path : str or Path
        Path to directory containing waveform CSV files
    scope_runs : list
        List of scope run numbers to load (e.g., [43, 44, 45])
    gate_threshold : float, optional
        Voltage threshold for detecting gate high state (default: 2.3)
    time_crop : tuple, optional
        (start_idx, end_idx) for cropping time series (default: (530, 865))
    waveform_of_run_mV : float, optional 
        Voltage of the actual run waveform in mV (default: 12)
    waveform_of_measurement_mV : float, optional
        Voltage used during waveform measurement in mV (default: 5)
    waveform_scale : float, optional
        Scale factor for waveform voltage (default: 20)
    gate_width_timing_int: float, optional
        Value that depends on the width of the gates of interest. For larger gates use a larger integer. 
        For gate width of 10 us use 10, for gate width of 7.5 us use 7.
    gate_offset : float, optional
        Offset to add to gate signal (default: 0.35)
    gate_scale : float, optional
        Scale factor for gate signal (default: 0.5)
    xlim : tuple, optional
        X-axis limits (default: (-25, 200))
    waveform_ylim : tuple, optional
        Y-axis limits for waveform. If None, auto-scales (default: None)
    delays_to_show : list, optional
        List of delay values to show in legend. If None, uses all.
    figsize : tuple, optional
        Figure size (default: (14, 6))
    array_index : int, optional
        For array-type moments (com, variance, etc.):
        0 = row direction, 1 = column direction (default: 0)
    show_gates : bool, optional
        Whether to show gate signals (default: True)
    show_fractional : bool, optional
        Whether to show fractional difference plot (default: True)
    
    Returns:
    --------
    fig : matplotlib figure object
    axes : tuple of axes objects
    time_shifted : list of shifted time arrays
    """
    from pathlib import Path
    import numpy as np
    import matplotlib.pyplot as plt
    
    waveform_path = Path(waveform_path)
    
    # Extract data from results
    delay_dict = results['delay_dict']
    delay_list = list(delay_dict.values())
    
    # Map moment_key to actual dictionary keys in results
    moment_key_map = {
        'intensities': ('intensities_g1', 'intensities_g2'),
        'com': ('com_g1', 'com_g2'),
        'variance': ('variance_g1', 'variance_g2'),
        'skewness': ('skewness_g1', 'skewness_g2'),
        'kurtosis': ('kurtosis_g1', 'kurtosis_g2'),
        'background': ('background_g1', 'background_g2')
    }
    
    if moment_key not in moment_key_map:
        raise ValueError(f"moment_key must be one of {list(moment_key_map.keys())}")
    
    g1_key, g2_key = moment_key_map[moment_key]
    moment_g1 = results[g1_key]
    moment_g2 = results[g2_key]
    
    # Load waveform data
    time = []
    waveform = []
    gates = []
    
    for run in scope_runs:
        file = waveform_path / f'RigolDS{run}.csv'
        if not file.exists():
            print(f"Warning: {file} not found, skipping...")
            continue
            
        extract = np.genfromtxt(fname=file, delimiter=',', dtype=float)
        time.append(extract[time_crop[0]:time_crop[1], 0] * 1e6)  # Convert to μs
        waveform.append(-(extract[time_crop[0]:time_crop[1], 1] * 
                         (waveform_of_run_mV / waveform_of_measurement_mV)) * waveform_scale)
        gates.append((extract[time_crop[0]:time_crop[1], 2] * gate_scale) + gate_offset)
    
    if len(time) == 0:
        raise ValueError("No waveform files found!")
    
    # Find gate center for time shifting
    gate_high = gates[0] > gate_threshold
    high_indices = np.where(gate_high)[0]
    
    if len(high_indices) == 0:
        raise ValueError(f"No gate signal found above threshold {gate_threshold}V")
    
    time_high = time[0][high_indices]
    t_center = np.mean(time_high[0:min(gate_width_timing_int, len(time_high))])
    
    print(f"Gate center time: {t_center:.2f} μs")
    
    # Shift all time arrays
    time_shifted = [t - t_center for t in time]
    
    # Extract moment differences
    runs = list(moment_g1.keys())
    
    # Define direction label
    direction_label = ['Row', 'Col'][array_index]
    
    # Array-type moments
    array_type_moments = ['com', 'variance', 'skewness', 'kurtosis']
    
    # Handle different moment types
    if moment_key in array_type_moments:
        # Array values - use specified index
        moment_diff = [moment_g2[r][array_index] - moment_g1[r][array_index] for r in runs]
        fra_moment_diff = [(moment_g2[r][array_index] - moment_g1[r][array_index]) / 
                          moment_g1[r][array_index] for r in runs]
        ylabel_suffix = f' ({moment_key.title()} - {direction_label})'
    else:
        # Scalar values
        moment_diff = [moment_g2[r] - moment_g1[r] for r in runs]
        fra_moment_diff = [(moment_g2[r] - moment_g1[r]) / moment_g1[r] for r in runs]
        ylabel_suffix = f' ({moment_key.title()})'
    
    # Create subplots
    if show_fractional:
        fig, (ax1_left, ax1_right) = plt.subplots(1, 2, figsize=figsize)
    else:
        fig, ax1_left = plt.subplots(1, 1, figsize=(figsize[0]//2, figsize[1]))
    
    # LEFT PLOT: Absolute difference with waveform
    ax1_left.plot(np.array(delay_list).astype(float), moment_diff, 'o', color='tab:red')
    ax1_left.set_xlabel('Delay Time (μs)', fontsize=14)
    ax1_left.set_ylabel(f'Difference (G2-G1){ylabel_suffix}', color='tab:red', fontsize=13)
    ax1_left.tick_params(axis='y', labelcolor='tab:red')
    ax1_left.set_xlim(xlim)
    
    # Overlay waveform on left plot
    ax2_left = ax1_left.twinx()
    ax2_left.plot(time_shifted[0], waveform[0], color='k', linewidth=1.5, label='Waveform')
    ax2_left.set_ylabel('Sample Voltage (V)', color='k', fontsize=14)
    if waveform_ylim is not None:
        ax2_left.set_ylim(waveform_ylim)
    
    # Overlay gates on left plot if requested
    if show_gates:
        ax3_left = ax1_left.twinx()
        #ax3_left.spines['right'].set_position(('outward', 60))
        
        # Determine which delays to show in legend
        if delays_to_show is None:
            delays_to_show = delay_list
        
        legend_labels = [f'{d} μs' for d in delays_to_show]
        
        for i in range(len(time_shifted)):
            ax3_left.plot(time_shifted[i], gates[i], alpha=0.7)
        
        #ax3_left.set_ylabel('Gate Signal (V)', fontsize=12)
        ax3_left.axis('off')
        ax3_left.legend(legend_labels, title='Delay Times', loc='lower right', fontsize=10)
    
    # RIGHT PLOT: Fractional difference with waveform (if requested)
    if show_fractional:
        ax1_right.plot(np.array(delay_list).astype(float), fra_moment_diff, 'o', color='tab:purple')
        ax1_right.set_xlabel('Delay Time (μs)', fontsize=14)
        ax1_right.set_ylabel(f'Frac. Diff. (G2-G1)/G1{ylabel_suffix}', color='tab:purple', fontsize=13)
        ax1_right.tick_params(axis='y', labelcolor='tab:purple')
        ax1_right.set_xlim(xlim)
        
        # Overlay waveform on right plot
        ax2_right = ax1_right.twinx()
        ax2_right.plot(time_shifted[0], waveform[0], color='k', linewidth=1.5)
        ax2_right.set_ylabel('Sample Voltage (V)', color='k', fontsize=14)
        if waveform_ylim is not None:
            ax2_right.set_ylim(waveform_ylim)
        
        # Overlay gates on right plot if requested
        if show_gates:
            ax3_right = ax1_right.twinx()
            #ax3_right.spines['right'].set_position(('outward', 60))
            
            for i in range(len(time_shifted)):
                ax3_right.plot(time_shifted[i], gates[i], alpha=0.7)
            
            #ax3_right.set_ylabel('Gate Signal (V)', fontsize=12)
            ax3_right.axis('off')
    plt.tight_layout()
    
    # Return appropriate axes
    if show_fractional and show_gates:
        return fig, (ax1_left, ax2_left, ax3_left, ax1_right, ax2_right, ax3_right), time_shifted
    elif show_fractional:
        return fig, (ax1_left, ax2_left, ax1_right, ax2_right), time_shifted
    elif show_gates:
        return fig, (ax1_left, ax2_left, ax3_left), time_shifted
    else:
        return fig, (ax1_left, ax2_left), time_shifted


def plot_waveform_with_moments_pund(results, moment_key,
                               waveform_path, scope_runs, gate_threshold=2.3,
                               time_crop=(530, 865), waveform_offset = 0.0019, waveform_of_run_mV=12, 
                               waveform_of_measurement_mV=5, waveform_scale=20,
                               gate_width_timing_int = 10, 
                               gate_offset=0.35, gate_scale=0.5,
                               xlim=(-25, 200), waveform_ylim=None,
                               delays_to_show=None, figsize=(14, 6), array_index=0,
                               show_gates=True, show_fractional=True):
    """
    Plot moment differences vs delay time with synchronized voltage waveform and gate signals.
    
    Parameters:
    -----------
    results : dict
        Output dictionary from process_scan_data containing all moments and delay_dict
    moment_key : str
        Key for moment to plot from results (e.g., 'intensities', 'com', 'variance')
        Valid keys: 'intensities', 'com', 'variance', 'skewness', 'kurtosis', 'background'
    waveform_path : str or Path
        Path to directory containing waveform CSV files
    scope_runs : list
        List of scope run numbers to load (e.g., [43, 44, 45])
    gate_threshold : float, optional
        Voltage threshold for detecting gate high state (default: 2.3)
    time_crop : tuple, optional
        (start_idx, end_idx) for cropping time series (default: (530, 865))
    waveform_of_run_mV : float, optional 
        Voltage of the actual run waveform in mV (default: 12)
    waveform_of_measurement_mV : float, optional
        Voltage used during waveform measurement in mV (default: 5)
    waveform_scale : float, optional
        Scale factor for waveform voltage (default: 20)
    gate_width_timing_int: float, optional
        Value that depends on the width of the gates of interest. For larger gates use a larger integer. 
        For gate width of 10 us use 10, for gate width of 7.5 us use 7.
    gate_offset : float, optional
        Offset to add to gate signal (default: 0.35)
    gate_scale : float, optional
        Scale factor for gate signal (default: 0.5)
    xlim : tuple, optional
        X-axis limits (default: (-25, 200))
    waveform_ylim : tuple, optional
        Y-axis limits for waveform. If None, auto-scales (default: None)
    delays_to_show : list, optional
        List of delay values to show in legend. If None, uses all.
    figsize : tuple, optional
        Figure size (default: (14, 6))
    array_index : int, optional
        For array-type moments (com, variance, etc.):
        0 = row direction, 1 = column direction (default: 0)
    show_gates : bool, optional
        Whether to show gate signals (default: True)
    show_fractional : bool, optional
        Whether to show fractional difference plot (default: True)
    
    Returns:
    --------
    fig : matplotlib figure object
    axes : tuple of axes objects
    time_shifted : list of shifted time arrays
    """
    from pathlib import Path
    import numpy as np
    import matplotlib.pyplot as plt
    
    waveform_path = Path(waveform_path)
    
    # Extract data from results
    delay_dict = results['delay_dict']
    delay_list = list(delay_dict.values())
    
    # Map moment_key to actual dictionary keys in results
    moment_key_map = {
        'intensities': ('intensities_g1', 'intensities_g2'),
        'com': ('com_g1', 'com_g2'),
        'variance': ('variance_g1', 'variance_g2'),
        'skewness': ('skewness_g1', 'skewness_g2'),
        'kurtosis': ('kurtosis_g1', 'kurtosis_g2'),
        'background': ('background_g1', 'background_g2')
    }
    
    if moment_key not in moment_key_map:
        raise ValueError(f"moment_key must be one of {list(moment_key_map.keys())}")
    
    g1_key, g2_key = moment_key_map[moment_key]
    moment_g1 = results[g1_key]
    moment_g2 = results[g2_key]
    
    # Load waveform data
    time = []
    waveform = []
    gates = []
    
    for run in scope_runs:
        file = waveform_path / f'RigolDS{run}.csv'
        if not file.exists():
            print(f"Warning: {file} not found, skipping...")
            continue
            
        extract = np.genfromtxt(fname=file, delimiter=',', dtype=float)
        time.append(extract[time_crop[0]:time_crop[1], 0] * 1e6)  # Convert to μs
        waveform.append((-(extract[time_crop[0]:time_crop[1], 2] * 
                         (waveform_of_run_mV / waveform_of_measurement_mV)) * waveform_scale)+waveform_offset)
        gates.append((extract[time_crop[0]:time_crop[1], 1] * gate_scale) + gate_offset)
    
    if len(time) == 0:
        raise ValueError("No waveform files found!")
    
    # Find gate center for time shifting
    gate_high = gates[0] > gate_threshold
    high_indices = np.where(gate_high)[0]
    
    if len(high_indices) == 0:
        raise ValueError(f"No gate signal found above threshold {gate_threshold}V")
    
    time_high = time[0][high_indices]
    t_center = np.mean(time_high[0:min(gate_width_timing_int, len(time_high))])
    
    print(f"Gate center time: {t_center:.2f} μs")
    
    # Shift all time arrays
    time_shifted = [t - t_center for t in time]
    
    # Extract moment differences
    runs = list(moment_g1.keys())
    
    # Define direction label
    direction_label = ['Row', 'Col'][array_index]
    
    # Array-type moments
    array_type_moments = ['com', 'variance', 'skewness', 'kurtosis']
    
    # Handle different moment types
    if moment_key in array_type_moments:
        # Array values - use specified index
        moment_diff = [moment_g2[r][array_index] - moment_g1[r][array_index] for r in runs]
        fra_moment_diff = [(moment_g2[r][array_index] - moment_g1[r][array_index]) / 
                          moment_g1[r][array_index] for r in runs]
        ylabel_suffix = f' ({moment_key.title()} - {direction_label})'
    else:
        # Scalar values
        moment_diff = [moment_g2[r] - moment_g1[r] for r in runs]
        fra_moment_diff = [(moment_g2[r] - moment_g1[r]) / moment_g1[r] for r in runs]
        ylabel_suffix = f' ({moment_key.title()})'
    
    # Create subplots
    if show_fractional:
        fig, (ax1_left, ax1_right) = plt.subplots(1, 2, figsize=figsize)
    else:
        fig, ax1_left = plt.subplots(1, 1, figsize=(figsize[0]//2, figsize[1]))
    
    # LEFT PLOT: Absolute difference with waveform
    ax1_left.plot(np.array(delay_list).astype(float), moment_diff, 'o', color='tab:red')
    ax1_left.set_xlabel('Delay Time (μs)', fontsize=14)
    ax1_left.set_ylabel(f'Difference (G2-G1){ylabel_suffix}', color='tab:red', fontsize=13)
    ax1_left.tick_params(axis='y', labelcolor='tab:red')
    ax1_left.set_xlim(xlim)
    
    # Overlay waveform on left plot
    ax2_left = ax1_left.twinx()
    ax2_left.plot(time_shifted[0], waveform[0], color='k', linewidth=1.5, label='Waveform')
    ax2_left.set_ylabel('Sample Voltage (V)', color='k', fontsize=14)
    if waveform_ylim is not None:
        ax2_left.set_ylim(waveform_ylim)
    
    # Overlay gates on left plot if requested
    if show_gates:
        ax3_left = ax1_left.twinx()
        #ax3_left.spines['right'].set_position(('outward', 60))
        
        # Determine which delays to show in legend
        if delays_to_show is None:
            delays_to_show = delay_list
        
        legend_labels = [f'{d} μs' for d in delays_to_show]
        
        for i in range(len(time_shifted)):
            ax3_left.plot(time_shifted[i], gates[i], alpha=0.7)
        
        #ax3_left.set_ylabel('Gate Signal (V)', fontsize=12)
        ax3_left.axis('off')
        ax3_left.legend(legend_labels, title='Delay Times', loc='lower right', fontsize=10)
    
    # RIGHT PLOT: Fractional difference with waveform (if requested)
    if show_fractional:
        ax1_right.plot(np.array(delay_list).astype(float), fra_moment_diff, 'o', color='tab:purple')
        ax1_right.set_xlabel('Delay Time (μs)', fontsize=14)
        ax1_right.set_ylabel(f'Frac. Diff. (G2-G1)/G1{ylabel_suffix}', color='tab:purple', fontsize=13)
        ax1_right.tick_params(axis='y', labelcolor='tab:purple')
        ax1_right.set_xlim(xlim)
        
        # Overlay waveform on right plot
        ax2_right = ax1_right.twinx()
        ax2_right.plot(time_shifted[0], waveform[0], color='k', linewidth=1.5)
        ax2_right.set_ylabel('Sample Voltage (V)', color='k', fontsize=14)
        if waveform_ylim is not None:
            ax2_right.set_ylim(waveform_ylim)
        
        # Overlay gates on right plot if requested
        if show_gates:
            ax3_right = ax1_right.twinx()
            #ax3_right.spines['right'].set_position(('outward', 60))
            
            for i in range(len(time_shifted)):
                ax3_right.plot(time_shifted[i], gates[i], alpha=0.7)
            
            #ax3_right.set_ylabel('Gate Signal (V)', fontsize=12)
            ax3_right.axis('off')
    plt.tight_layout()
    
    # Return appropriate axes
    if show_fractional and show_gates:
        return fig, (ax1_left, ax2_left, ax3_left, ax1_right, ax2_right, ax3_right), time_shifted
    elif show_fractional:
        return fig, (ax1_left, ax2_left, ax1_right, ax2_right), time_shifted
    elif show_gates:
        return fig, (ax1_left, ax2_left, ax3_left), time_shifted
    else:
        return fig, (ax1_left, ax2_left), time_shifted

def plot_waveform_and_current_with_moments(results, moment_key,
                               gate_path, scope_runs, gate_threshold=2.3,
                               time_crop=(530, 865), waveform_of_run_mV=12, 
                               waveform_of_measurement_mV=5, waveform_scale=20,
                               gate_width_timing_int = 10, 
                               gate_offset=0.35, gate_scale=0.5,
                               current_path=None, 
                               current_time_crop=None, current_waveform_voltage=0.24,
                               current_waveform_freq = 3.0,
                               current_scale=1.0, current_offset=0.0,
                               xlim=(-25, 200), waveform_ylim=None, current_ylim=None,
                               delays_to_show=None, figsize=(14, 6), array_index=0,
                               show_gates=True, show_fractional=True, show_current=True,
                               align_waveforms=True, n_peaks=8):
    """
    Plot moment differences vs delay time with synchronized voltage waveform, gate signals, and current response.
    
    Parameters:
    -----------
    results : dict
        Output dictionary from process_scan_data containing all moments and delay_dict
    moment_key : str
        Key for moment to plot from results (e.g., 'intensities', 'com', 'variance')
        Valid keys: 'intensities', 'com', 'variance', 'skewness', 'kurtosis', 'background'
    gate_path : str or Path
        Path to directory containing waveform CSV files
    scope_runs : list
        List of scope run numbers to load (e.g., [43, 44, 45])
    gate_threshold : float, optional
        Voltage threshold for detecting gate high state (default: 2.3)
    time_crop : tuple, optional
        (start_idx, end_idx) for cropping time series (default: (530, 865))
    waveform_of_run_mV : float, optional 
        Voltage of the actual run waveform in mV (default: 12)
    waveform_of_measurement_mV : float, optional
        Voltage used during waveform measurement in mV (default: 5)
    waveform_scale : float, optional
        Scale factor for waveform voltage (default: 20)
    gate_width_timing_int: float, optional
        Value that depends on the width of the gates of interest. For larger gates use a larger integer. 
        For gate width of 10 us use 10, for gate width of 7.5 us use 7.
    gate_offset : float, optional
        Offset to add to gate signal (default: 0.35)
    gate_scale : float, optional
        Scale factor for gate signal (default: 0.5)
    current_path : str or Path, optional
        Path to directory containing current measurement CSV file (default: None)
    current_time_crop : tuple, optional
        (start_idx, end_idx) for cropping current time series. If None, uses time_crop (default: None)
    current_waveform_voltage : float, optional
        Voltage of the waveform during current measurement in V (default: 0.24)
    current_waveform_freq : float, optional
        Frequency of current waveform in kHz (default: 3.0)
    current_scale : float, optional
        Scale factor for current signal (default: 1.0)
    current_offset : float, optional
        Offset to add to current signal (default: 0.0)
    xlim : tuple, optional
        X-axis limits (default: (-25, 200))
    waveform_ylim : tuple, optional
        Y-axis limits for waveform. If None, auto-scales (default: None)
    current_ylim : tuple, optional
        Y-axis limits for current. If None, auto-scales (default: None)
    delays_to_show : list, optional
        List of delay values to show in legend. If None, uses all.
    figsize : tuple, optional
        Figure size (default: (14, 6))
    array_index : int, optional
        For array-type moments (com, variance, etc.):
        0 = row direction, 1 = column direction (default: 0)
    show_gates : bool, optional
        Whether to show gate signals (default: True)
    show_fractional : bool, optional
        Whether to show fractional difference plot (default: True)
    show_current : bool, optional
        Whether to show current response (default: True)
    align_waveforms : bool, optional
        Whether to align waveforms using peak detection (default: True)
    n_peaks : int, optional
        Number of peaks to use for alignment (default: 4)
    
    Returns:
    --------
    fig : matplotlib figure object
    axes : tuple of axes objects
    time_shifted : list of shifted time arrays
    current_time_shifted : shifted time array for current (if show_current=True)
    """
    from pathlib import Path
    import numpy as np
    import matplotlib.pyplot as plt
    
    gate_path = Path(gate_path)
    
    # Extract data from results
    delay_dict = results['delay_dict']
    delay_list = list(delay_dict.values())
    
    # Map moment_key to actual dictionary keys in results
    moment_key_map = {
        'intensities': ('intensities_g1', 'intensities_g2'),
        'com': ('com_g1', 'com_g2'),
        'variance': ('variance_g1', 'variance_g2'),
        'skewness': ('skewness_g1', 'skewness_g2'),
        'kurtosis': ('kurtosis_g1', 'kurtosis_g2'),
        'background': ('background_g1', 'background_g2')
    }
    
    if moment_key not in moment_key_map:
        raise ValueError(f"moment_key must be one of {list(moment_key_map.keys())}")
    
    g1_key, g2_key = moment_key_map[moment_key]
    moment_g1 = results[g1_key]
    moment_g2 = results[g2_key]
    
    # Load waveform data (gate files)
    time = []
    waveform = []
    gates = []
    
    for run in scope_runs:
        file = gate_path / f'RigolDS{run}.csv'
        if not file.exists():
            print(f"Warning: {file} not found, skipping...")
            continue
            
        extract = np.genfromtxt(fname=file, delimiter=',', dtype=float)
        time.append(extract[time_crop[0]:time_crop[1], 0] * 1e6)  # Convert to μs
        waveform.append(-(extract[time_crop[0]:time_crop[1], 1] * 
                         (waveform_of_run_mV / waveform_of_measurement_mV)) * waveform_scale)
        gates.append((extract[time_crop[0]:time_crop[1], 2] * gate_scale) + gate_offset)
    
    if len(time) == 0:
        raise ValueError("No waveform files found!")
    
    # Load current measurement data if requested
    current_time = None
    current_waveform_data = None
    current_response = None
    current_time_shifted = None
    waveform_time_shift = 0.0
    
    if show_current and current_path is not None and current_waveform_voltage is not None and current_waveform_freq is not None:
        current_path = Path(current_path)
        pattern = f'BTO_A2_triangle_f{current_waveform_freq}kHz_amp{current_waveform_voltage}V_*.csv'
        
        # Find files matching the pattern
        matching_files = list(current_path.glob(pattern))
        
        if len(matching_files) > 0:
            current_file = matching_files[0]  # Use the first matching file
            print(f"Found {len(matching_files)} matching file(s): {[f.name for f in matching_files]}")
            print(f"Using: {current_file}")
            
            current_extract = np.genfromtxt(fname=current_file, delimiter=',', dtype=float)
            print(f"Loaded current extract shape: {current_extract.shape}")
            
            # Handle cropping - use all data if current_time_crop is None
            if current_time_crop is not None:
                start_idx = current_time_crop[0] if current_time_crop[0] is not None else 0
                end_idx = current_time_crop[1] if current_time_crop[1] is not None else len(current_extract)
                # Skip first row (index 0) to avoid NaN
                current_time = current_extract[start_idx+1:end_idx, 0] * 1e6
                current_response = (current_extract[start_idx+1:end_idx, 1] * current_scale) + current_offset
                current_waveform_data = current_extract[start_idx+1:end_idx, 2]
            else:
                # Use all data - no cropping, but skip first row
                current_time = current_extract[1:, 0] * 1e6  # Skip index 0
                current_response = ((current_extract[1:, 1] * current_scale) + current_offset) 
                current_waveform_data = current_extract[1:, 2]
            
            print(f"Current response shape: {current_response.shape}")
            print(f"Current time range: {current_time[0]:.2f} to {current_time[-1]:.2f} μs")
            print(f"Current response range: {np.min(current_response):.2e} to {np.max(current_response):.2e} A")
            print(f"show_current = {show_current}")
            
            #print(f"Current data shape: {current_response.shape}, time range: {current_time[0]:.2f} to {current_time[-1]:.2f} μs")
            
            # Align waveforms using peak detection
            if align_waveforms:
                # Scale both waveforms for comparison
                gate_waveform_scaled = waveform[0]
                current_waveform_scaled = -(current_waveform_data / current_waveform_voltage) * waveform_scale
                
                # Find top n peaks in both waveforms
                idx_top_gate = np.argsort(gate_waveform_scaled)[-n_peaks:][::-1]
                idx_top_current = np.argsort(current_waveform_scaled)[-n_peaks:][::-1]
                
                # Get time values at peaks
                time_top_gate = time[0][idx_top_gate]
                time_top_current = current_time[idx_top_current]
                
                # Calculate time shift based on mean of peak times
                #waveform_time_shift = np.mean(time_top_gate) - np.mean(time_top_current)
                print(f"rigol time index: {time[0][np.min(idx_top_gate)]}")
                print(f"current time index: {current_time[np.min(idx_top_current)]}")
                waveform_time_shift = (time[0][np.min(idx_top_gate)] - current_time[np.min(idx_top_current)]) + 100
                #time_shift = time_r[np.min(idx_top4_r)] - time[np.min(idx_top4)]

                
                print(f"Waveform alignment: {n_peaks} peaks used")
                print(f"Gate waveform peak times: {time_top_gate}")
                print(f"Current waveform peak times: {time_top_current}")
                print(f"Waveform time shift applied: {waveform_time_shift:.2f} μs")
            
            # Apply waveform alignment shift to current time
            current_time = current_time + waveform_time_shift + (((1/current_waveform_freq)*1e3)/2)
            
            print(f"Loaded current data from {current_file}")
        else:
            print(f"Warning: Current file {current_file} not found, skipping current plot...")
            show_current = False
    
    # Find gate center for time shifting
    gate_high = gates[0] > gate_threshold
    high_indices = np.where(gate_high)[0]
    
    if len(high_indices) == 0:
        raise ValueError(f"No gate signal found above threshold {gate_threshold}V")
    
    time_high = time[0][high_indices]
    t_center = np.mean(time_high[0:min(gate_width_timing_int, len(time_high))])
    
    print(f"Gate center time: {t_center:.2f} μs")
    
    # Shift all time arrays relative to gate center
    time_shifted = [t - t_center for t in time]
    
    # Shift current time array if it exists (already includes waveform alignment)
    if current_time is not None:
       current_time_shifted = current_time - t_center
    
    # Extract moment differences
    runs = list(moment_g1.keys())
    
    # Define direction label
    direction_label = ['Row', 'Col'][array_index]
    
    # Array-type moments
    array_type_moments = ['com', 'variance', 'skewness', 'kurtosis']
    
    # Handle different moment types
    if moment_key in array_type_moments:
        # Array values - use specified index
        moment_diff = [moment_g2[r][array_index] - moment_g1[r][array_index] for r in runs]
        fra_moment_diff = [(moment_g2[r][array_index] - moment_g1[r][array_index]) / 
                          moment_g1[r][array_index] for r in runs]
        ylabel_suffix = f' ({moment_key.title()} - {direction_label})'
    else:
        # Scalar values
        moment_diff = [moment_g2[r] - moment_g1[r] for r in runs]
        fra_moment_diff = [(moment_g2[r] - moment_g1[r]) / moment_g1[r] for r in runs]
        ylabel_suffix = f' ({moment_key.title()})'
    
    # Create subplots
    if show_fractional:
        fig, (ax1_left, ax1_right) = plt.subplots(1, 2, figsize=figsize)
    else:
        fig, ax1_left = plt.subplots(1, 1, figsize=(figsize[0]//2, figsize[1]))
    
    # LEFT PLOT: Absolute difference with waveform
    ax1_left.plot(np.array(delay_list).astype(float), moment_diff, 'o', color='tab:red')
    ax1_left.set_xlabel('Delay Time (μs)', fontsize=14)
    ax1_left.set_ylabel(f'Difference (G2-G1){ylabel_suffix}', color='tab:red', fontsize=13)
    ax1_left.tick_params(axis='y', labelcolor='tab:red')
    ax1_left.set_xlim(xlim)
    
   # Overlay gates on left plot FIRST (so they're behind other elements)
    if show_gates:
        ax3_left = ax1_left.twinx()
        ax3_left.spines['right'].set_visible(False)  # Hide this axis spine
        
        # Determine which delays to show in legend
        if delays_to_show is None:
            delays_to_show_set = set(delay_list)
        else:
            # Convert delays_to_show to strings to match delay_list format
            delays_to_show_set = set(str(int(d)) if d == int(d) else str(d) for d in delays_to_show)

        # Define colors for the gate plots
        gate_colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:orange', 'tab:orange', 
                   'tab:orange', 'tab:green', 'tab:green', 'tab:green']
        
        print(f"delay_list: {delay_list}")
        print(f"delays_to_show_set: {delays_to_show_set}")
        
        # Plot all gates
        for i in range(len(time_shifted)):
            delay_value = delay_list[i]
            
            # Check if this delay should be in the legend
            if delay_value in delays_to_show_set:
                label = f'{delay_value} μs'
                print(f"Adding to legend: {delay_value}")
            else:
                label = None  # Don't show in legend
            
            ax3_left.plot(time_shifted[i], gates[i], alpha=0.5, color = gate_colors[i], label=label)

        # Add dummy plots for delays in delays_to_show that we don't have waveform data for
        # This ensures they appear in the legend
        for delay_value in delay_list:
            if delay_value in delays_to_show_set:
                # Check if this delay was already plotted
                delay_index = delay_list.index(delay_value)
                if delay_index >= len(time_shifted):
                    # Add a dummy invisible line just for the legend
                    ax3_left.plot([], [], alpha=0.5, label=f'{delay_value} μs')
        
        ax3_left.legend(title='Delay Times', loc='upper right', fontsize=10)
        ax3_left.tick_params(axis='y', which='both', left=False, right=False, labelleft=False, labelright=False)
    
    # Overlay waveform on left plot (SECOND position - rightmost)
    ax2_left = ax1_left.twinx()
    ax2_left.plot(time_shifted[0], waveform[0], color='k', linewidth=1.5, alpha=0.7)
    
    # Overlay current waveform if available
    if show_current and current_waveform_data is not None:
        # Scale current waveform to match gate waveform
        current_waveform_scaled = -(current_waveform_data / current_waveform_voltage) * waveform_scale
    
    ax2_left.set_ylabel('Sample Voltage (V)', color='k', fontsize=14)
    if waveform_ylim is not None:
        ax2_left.set_ylim(waveform_ylim)
    
    # Add current response if available (THIRD position - move it outward)
    if show_current and current_response is not None:
        ax4_left = ax1_left.twinx()
        ax4_left.spines['right'].set_position(('outward', 60))
        
        ax4_left.plot(current_time_shifted, current_response, 
                     color='tab:green', linewidth=1.5)
        ax4_left.set_ylabel('Current Response (A)', color='tab:green', fontsize=12)
        ax4_left.tick_params(axis='y', labelcolor='tab:green')
        if current_ylim is not None:
            ax4_left.set_ylim(current_ylim)
    
    # RIGHT PLOT: Fractional difference with waveform (if requested)
    if show_fractional:
        ax1_right.plot(np.array(delay_list).astype(float), fra_moment_diff, 'o', color='tab:purple')
        ax1_right.set_xlabel('Delay Time (μs)', fontsize=14)
        ax1_right.set_ylabel(f'Frac. Diff. (G2-G1)/G1{ylabel_suffix}', color='tab:purple', fontsize=13)
        ax1_right.tick_params(axis='y', labelcolor='tab:purple')
        ax1_right.set_xlim(xlim)
        
        # Overlay waveform on right plot
        ax2_right = ax1_right.twinx()
        ax2_right.plot(time_shifted[0], waveform[0], color='k', linewidth=1.5, alpha=0.7)
        
        # Overlay current waveform if available
        if show_current and current_waveform_data is not None:
            current_waveform_scaled = -(current_waveform_data / current_waveform_voltage) * waveform_scale
        #     ax2_right.plot(current_time_shifted, current_waveform_scaled, 
        #                  color='gray', linewidth=1.5, linestyle='--',
        #                  label='Waveform (Current)', alpha=0.7)
        
        ax2_right.set_ylabel('Sample Voltage (V)', color='k', fontsize=14)
        if waveform_ylim is not None:
            ax2_right.set_ylim(waveform_ylim)
        #ax2_right.legend(loc='upper left', fontsize=10)
        
        # Overlay gates on right plot if requested
        if show_gates:
            ax3_right = ax1_right.twinx()
            ax3_right.spines['right'].set_position(('outward', 60))
            
            for i in range(len(time_shifted)):
                ax3_right.plot(time_shifted[i], gates[i], alpha=0.5)
            
            ax3_right.set_ylabel('Gate Signal (V)', fontsize=12, color='tab:blue')
            ax3_right.tick_params(axis='y', labelcolor='tab:blue')
        
        # Add current response on right plot if available
        if show_current and current_response is not None:
            ax4_right = ax1_right.twinx()
            if show_gates:
                ax4_right.spines['right'].set_position(('outward', 120))
            else:
                ax4_right.spines['right'].set_position(('outward', 60))
            
            ax4_right.plot(current_time_shifted, current_response, 
                         color='tab:green', linewidth=1.5) #, label='Current Response'
            ax4_right.set_ylabel('Current Response (A)', color='tab:green', fontsize=12)
            ax4_right.tick_params(axis='y', labelcolor='tab:green')
            if current_ylim is not None:
                ax4_right.set_ylim(current_ylim)
            #ax4_right.legend(loc='lower left', fontsize=10)
    
    plt.tight_layout()
    
    # Return appropriate axes based on what's shown
    axes_list = [ax1_left, ax2_left]
    
    if show_gates:
        axes_list.append(ax3_left)
    if show_current and current_response is not None:
        axes_list.append(ax4_left)
    
    if show_fractional:
        axes_list.extend([ax1_right, ax2_right])
        if show_gates:
            axes_list.append(ax3_right)
        if show_current and current_response is not None:
            axes_list.append(ax4_right)
    
    # Always return 4 values for consistency
    return fig, tuple(axes_list), time_shifted, current_time_shifted

In [ ]:
#test run
#filenames = h5_loc.glob('run*_data_000001.h5')
good_runs = [673]

#voltage ramp later scans 600+
roi = [190, 260, 525, 675] # bto and ctr peaks
#roi = [190, 260, 543, 550] #ctr peak ONLY
#roi = [190, 260, 577, 635] #bto peak ONLY


background_roi = [190, 260, 900, 1000]  # Can play around with this, always check
lineout_range = (30, 40) #eta scans = 70-90

for file in good_runs:
    filenames = h5_loc.glob(f'run{file}*_data_000001.h5')
    print(filenames)
    
    for file in filenames:
        info = parse_filename(file, fields=['run', 'voltage', 'delay', 'eta']) 
        
        # Extract parsed values
        run = info['run']
        voltage = info['voltage']
        delay = info['delay']
        #eta = info['eta']
        
        h5 = h5py.File(file, 'r')
        data_raw = h5['entry/data/data'][:]
        h5.close()
    
    # hot pixel removal
    data = np.zeros_like(data_raw, dtype=float)
    for i in range(data_raw.shape[0]):
        #data[i] = remove_hot_pixels(data_raw[i], simple_threshold=4e5, neighbor_ratio=5)
        data[i] = remove_hot_pixels(data_raw[i])

    # Visualize ROIs (optional, for first run)
    if run == good_runs[0]:
         visualize_rois(data[0], roi, background_roi)
    
    # Calculate with background subtraction
    moments_g1 = calc_moments(data[0], roi, background_roi)
    moments_g2 = calc_moments(data[1], roi, background_roi)   

    #print(f"  G1 aCOM: {moments_g1['com_abs']}")

    print_moments(moments_g1, moments_g2, keys = ['background', 'com_abs', 'variance', 'kurtosis'], run=run)
    
    # Plot difference image with COM
    fig, ax = plot_diff_image(data, roi, moments_g1, moments_g2, run=run, voltage=voltage, delay=delay,
                          vmin=-50, vmax=50, show_com = True)
    
    lineout_1=np.nansum(data[0][roi[0]+lineout_range[0]:roi[0]+lineout_range[1],roi[2]:roi[3]],axis=0) #lineout of G1 calculating the mean = sum/total inputs, BTO and CTR
    lineout_2=np.nansum(data[1][roi[0]+lineout_range[0]:roi[0]+lineout_range[1],roi[2]:roi[3]],axis=0) #lineout of G2 calculating the mean = sum/total inputs, BTO and CTR
    
    fig, ax = plot_lineouts(lineout_1, lineout_2, moments_g1, moments_g2, run=run, voltage=voltage, delay=delay, eta = None, figsize = (28,5), show_com = True)

In [ ]:
#define roi's to use for comparing runs
both_roi = [190, 260, 525, 675] # bto and ctr peaks
bto_roi = [190, 260, 577, 635] #bto peak ONLY
ctr_roi = [190, 260, 543, 550] #ctr peak only
background_roi = [190, 260, 900, 1000]
exclude_ctr = (543,550)

In [ ]:
# VOLTAGE RAMP SCANS
# Basic usage of process_scan_data, plots difference and lineouts
# can adjust things like lineout range, diff_immage_kwargs (vmin and vmax), lineout_kwargs like figsize...
# can run for other plots but without plotting difference and lineout by using plot_diff_image_flag = Flase, and plot_lineouts_flag = False

#note that from the AWG this was 0 V and peak neg gate locations, given the amp inversion the sample was actually seeing gates at 0 V and peak pos gate locations
#therefore voltages are positve

results = process_scan_data(
    h5_loc=h5_loc,
    roi=both_roi,  
    #exclude_col_range = exclude_ctr,
    run_range=list(range(687, 689)) + list(range(690,703)) + [806],   #768, 782
    #run_range = [768],
    plot_diff_image_flag = False,
    diff_image_kwargs = {'vmin': -100, 'vmax': 100},
    plot_lineouts_flag = False,
    show_com_on_diff_image = False,
    show_com_on_lineouts=False,
)


# #Access results
voltage_list = list(results['voltage_dict'].values())
delay_list = list(results['delay_dict'].values())
intensity_g1_list = list(results['intensities_g1'].values())
intensity_g2_list = list(results['intensities_g2'].values())
# com_1_list = np.array(list(results['com_g1'].values()))
# com_2_list = np.array(list(results['com_g2'].values()))  
# var_1_list = np.array(list(results['variance_g1'].values()))
# var_2_list = np.array(list(results['variance_g2'].values())) 
# skew_1_list = np.array(list(results['skewness_g1'].values()))
# skew_2_list = np.array(list(results['skewness_g2'].values())) 
# kurt_1_list = np.array(list(results['kurtosis_g1'].values()))
# kurt_2_list = np.array(list(results['kurtosis_g2'].values()))

# # # Plot comparison, gate 1 G2 seperate, G2-G1, G2-G1/G1
fig, ax = plot_comparison_two(voltage_list, intensity_g1_list, intensity_g2_list,
                         xlabel='Voltage (V)',
                         ylabel='Intensity',figsize=(12,5))

# fig, ax = plot_comparison(voltage_list, com_1_list[:,1], com_2_list[:,1],
#                          xlabel='Voltage (V)',
#                          ylabel='COM Row Position')

# fig, ax = plot_comparison(voltage_list, var_1_list[:,1], var_2_list[:,1],
#                          xlabel='Voltage (V)',
#                          ylabel='Var Row Position')

# fig, ax = plot_comparison(voltage_list, skew_1_list[:,1], skew_2_list[:,1],
#                          xlabel='Voltage (V)',
#                          ylabel='Skew Row Position')

# fig, ax = plot_comparison(voltage_list, kurt_1_list[:,1], kurt_2_list[:,1],
#                          xlabel='Voltage (V)',
#                          ylabel='Kurt Row Position')

In [ ]:
# Animate lineouts vs delay (all data)
widgets = animate_lineouts(
    results,
    x_param='voltage',
    roi=ctr_roi,  # Needs ROI to know the offset
    #xlim=(620, 700),  # Show only pixels 620-700
    ylim=(0, 400000), #both = (0, 400000), bto = (0, 50000), ctr = (0,400000)
    show_com = True,
    ylim_diff = (-40000,300000), #both = (-40000,300000), bto = (-5000, 15000), ctr = (-40000,300000)
    ylim_fra_diff = (-0.5, 2), #both = (-0.5, 2), bto = (-0.2, 0.6), ctr = (-0.5, 2)
    use_absolute_pixels=True,  # Default
    interval=500,
    #save_path = save_path / 'bto001_peak002_ctr_roi_voltage_scan_triangle_lineout_animation.html',
    #save_html=True
)

widgets = animate_lineouts(
    results,
    x_param='voltage',
    roi=ctr_roi,  # Needs ROI to know the offset
    #xlim=(620, 700),  # Show only pixels 620-700
    ylim=(0, 400000), #both = (0, 400000), bto = (0, 50000), ctr = (0,400000)
    show_com = True,
    ylim_diff = (-40000,300000), #both = (-40000,300000), bto = (-5000, 15000), ctr = (-40000,300000)
    ylim_fra_diff = (-0.5, 2), #both = (-0.5, 2), bto = (-0.2, 0.6), ctr = (-0.5, 2)
    use_absolute_pixels=True,  # Default
    interval=500,
    #save_path = save_path / 'bto001_peak002_ctr_roi_voltage_scan_triangle_lineout_animation.gif',
    save_html=False
)

In [ ]:
widgets = animate_diff_images(
    results,
    x_param='voltage',
    roi=ctr_roi,
    xlim = (500, 750),
    vmin=-8000,
    vmax=8000,
    show_com=False,
    interval=750,
    #save_path = save_path / 'bto001_peak002_eta_roi_eta_scan_triangle_difference_animation_slower_logscale.gif',
    save_html=False,
    log_scale = True,
    linthresh = 10
)